## Lab 2: Photon Polarization States with QuTiP

In this lab we learn a specific computational framework called QuTiP, or the QUantum Toolbox In Python, released first in 2011. We use the interactive python computing environment provided by Jupyter Lab. We also need to import numpy package for doing math and manipulating arrays. 

In [1]:
import numpy as np
from qutip import basis

### 1. Basis Vectors for Photon Polarization States

Photon polarization states are represented by 2D complex vectors. We can use the most fundamental object Qobj in QuTiP, whose instances are matrices or vectors implemented efficiently in Python, to represent such states. First, basis($N$,$i$) function, which takes dimension $N$ and index $i$ as its arguments, creates a basis vector.

For photon polarization, the basis states can be chosen as horizontal $\vert H \rangle$ and vertical $\vert V \rangle$ polarization states. 

In [2]:
H = basis(2,0)
V = basis(2,1)

One can define alternative linear polarized basis vectors $\vert + \rangle$ and $\vert - \rangle$

In [3]:
P45 = 1 / np.sqrt(2) * (H + V)
M45 = 1 / np.sqrt(2) * (H - V)

and circularly polarized basis vectors

In [4]:
L = 1 / np.sqrt(2) * (H - 1.j * V)
R = 1 / np.sqrt(2) * (H + 1.j * V)

### 2. Inner Products

Inner products such as $\langle H \vert + \rangle$ can be calculated by converting the ket $\vert H \rangle$ to the bra $\langle + \vert$ using the .dag() method, which is defined for all instances of Qobj. 

In [5]:
print(H.dag() * P45)

(0.7071067811865475+0j)


Alternatively, we can use function overlap. For example, we calculate $\langle H \vert + \rangle$ by 

In [6]:
H.overlap(P45)

(0.7071067811865475+0j)

Note that the order of the bra and ket matters. 

In [7]:
print(V.overlap(R)) # <V|R>
print(R.overlap(V)) # <R|V>

0.7071067811865475j
-0.7071067811865475j


### 3. Projection Operators

We can also use the .dag() method to create projection operators. 

In [8]:
Ph = H * H.dag()
Pv = V * V.dag()

For example, $P_V \vert + \rangle$ represents the photon state that would result from passing light polarized at $+45^\circ$ through a vertical analyzer. 

In [9]:
print(Pv * P45)

Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.        ]
 [0.70710678]]


### 4. Quantum Probability Measurement

In general, we need to normalize the states. 

In [10]:
def probability_amplitude(s1, s2):
    return s1.unit().overlap(s2.unit())

def measurement_probability(s1, s2):
    return abs(probability_amplitude(s1.unit(), s2.unit()))**2

print(probability_amplitude(V, R))
print(measurement_probability(V, R))

0.7071067811865476j
0.5000000000000001


### 5. Quantum Key Distribution Using QuTiP

#### 5.1. Creating a Photon Class

Class is the key feature of object-oriented programming. A class is a structure for representing an object and the operations that can be performed on the object.

In Python a class can contain attributes (variables) and methods (functions). A class is defined almost like a function, but using the class keyword, and the class definition usually contains a number of class method definitions (a function in a class).

Each class method should have an argument self as its first argument. This object is a self-reference.

Some class method names have special meaning, see, for example, https://docs.python.org/3/reference/datamodel.html#special-method-names.

In [11]:
class Photon:
    ''' Photon class that can be encoded with either one of the two modes for QKD. '''
    def __init__(self, state):
        self._state = state
        
    def measure(self, m_basis):
        ''' All information one can read is through measurement, which may change the state. '''        
        if np.random.rand() < measurement_probability(self._state, m_basis[0]):
            self._state = m_basis[0]
            return 0
        else:
            self._state = m_basis[1]
            return 1
    
    def __repr__(self):
        ''' A method that is invoked when a simple string representation of the class is needed, 
            as for example when printed. '''
        return("{}, {}".format(self._state[0], self._state[1]))

In [12]:
# Encoding modes
basisHV = [H, V]
basisPM45 = [P45, M45]
modes = [basisHV, basisPM45]

In [13]:
# To create a new instance of a Photon class
p = Photon(P45)
print(p)

[0.70710678+0.j], [0.70710678+0.j]


In [14]:
# To invoke the measure class method in the class instance
print(p.measure(basisHV))
print(p)

0
[1.+0.j], [0.+0.j]


Notice that the state was randomly projected to $\vert H \rangle$ or $\vert V \rangle$, each with a 50% probability.

#### 5.2. Quantum Key Distribution

* Goal: To send encrypted messages that cannot be understood by anyone but the designated recipient. 
* Message is a whole number $m$, e.g., represented by the dots and dashes of Morse code as ones and zeros. 
* An encryption is a function $f$: $m \longrightarrow f(m)$, agreed on between Alice (sender) and Bob (recipient) but unknown to Eve (a possible eavesdropper). 
* Problem: If the same encryption is used many times, Eve can usually deduce the nature of the encryption and read the messages (by, e.g., frequency analysis). 

__Classical solution:__ Let the encryption depend on a frequently changed key, which can be regarded as another whole number $k$. The encrypted message is now $f(m, k)$. For example, $f(m,k) = km$. (In this case, the time Eve needs to factorize the product $km$ grows faster than any power of $km$.)
* New problem: Alice and Bob must frequently exchange messages to establish new keys, and these new messages too may be intercepted by Eve. 

__Quantum solution:__ _It is not possible to measure any quantity without changing an unknown state vector to one in which that quantity has some definite value._ 

| Mode   |  Binary digit     |  Angle value $\zeta$ |
|:----------:|:-------------:|:------:|
|  I | 0 | 0 | 
|    | 1 | $\pi/2$ | 
| II | 0 | $\pi/4$ | 
|    | 1 | $3\pi/4$ |

#### 5.3. The BB84 Protocol

<center>
<img src="Lab02_bb84.png" width="600">
</center>

* Alice sends the key to Bob as a sequence of linearly polarized photons with polarization vectors of the form $\vec{e} = (\cos \zeta, \sin \zeta)$, where $\zeta$ are various angles.  
* Alice represents ones and zeros by values of $\zeta$ in either one of the two modes. 
* Finally, Alice and Bob communicate over a classical, possibly public channel to compare their choices of basis for each bit. The bits for which Alice and Bob have used different bases are discarded.

In [15]:
def bit_generate(nb = 1):
    ''' Random initialize a bit, i.e., 0 or 1. '''
    return np.random.randint(0,2,nb)

# Alice's keys, randomly initialized.
nbits = 1000
Alice_bits = bit_generate(nbits)

In [16]:
def mode_select():
    ''' Randomly choose between mode HV and mode PM45. '''
    return np.random.randint(0,2)

# Alice: prepare photons to be tranmitted.
Alice_modes = []
Alice_prepared = []
for i in range(len(Alice_bits)):
    mode = mode_select()
    Alice_modes.append(mode)
    Alice_prepared.append(Photon(modes[mode][Alice_bits[i]]))

In the ideal case, Alice's list of photons is sent to Bob without interception. Bob randomly selects modes to measure the photons, then compares the sequence of modes with Alice. After sifting the list of keys, Alice and Bob obtain the same sequence, even though they have not communicated on what the keys are. 

In [17]:
Bob_received = Alice_prepared.copy()
Alice_prepared.clear()

In [18]:
# Bob randomly selects modes to measure the photons.
Bob_modes = []
Bob_bits = []
for p in Bob_received:
    mode = mode_select()
    Bob_modes.append(mode)
    Bob_bits.append(p.measure(modes[mode]))

In [19]:
# Alice and Bob compare their sequence of modes and sift the list of keys. 
keys = []
for am,bm,ab,bb in zip(Alice_modes,Bob_modes,Alice_bits,Bob_bits):
    if (am == bm): # compared modes through a classical channel
        assert ab == bb
        keys.append(ab)

In [20]:
print ('Sifted keys:', len(keys))
print(keys)

Sifted keys: 486
[np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int3

---

## Lab 2 Exercises

### E1. Eve's Strategy

Suppose Eve is a possible eavesdropper in the BB84 protocol. She tries to intercept the photons sent by Alice, measure their polarizations, and then send substitute photons with these polarizations on to Bob. But Eve, like Bob, does not know the mode that Alice is using in choosing each photon polarization. So there is only a 50% chance that the substitute photon sent by Eve to Bob will have the same polarization that it had when it was sent by Alice. 

When Alice and Bob compare notes, they identify the photons that had been sent when Alice and Bob had by chance being used the same modes. Eve, too, may learn this information, but by then it is too late. If Eve had used a different mode, there is still a 50% chance that Bob would have observed the same polarization that had been sent by Alice.

Simulate Eve's presence by insert a piece of code that realizes the following actions: Eve intercepts the photons sent by Alice, measures their polarizations, and then sends substitute photons with these polarizations on to Bob. Repeat key sifting between Alice and Bob. Show that, overall, Alice and Bob had 25% of the binary digits in the key that do not match; thus, they can detect Eve’s intervention by comparing a part of the key. 

Add Eve logic: Eve randomly selects modes to measure the photons sent by Alice, the sends the measured photons to Bob.

In [21]:
# Alice: prepare photons to be tranmitted.
Alice_modes = []
Alice_prepared = []
for i in range(len(Alice_bits)):
    mode = mode_select()
    Alice_modes.append(mode)
    Alice_prepared.append(Photon(modes[mode][Alice_bits[i]]))

# Eve receives the photons, assume no environmental noise
Eve_received = Alice_prepared.copy()
Alice_prepared.clear()

# Eve randomly selects modes to measure the photons.
Eve_modes = []
Eve_bits = []
Eve_prepared = []
for p in Eve_received:
    mode = mode_select()
    Eve_modes.append(mode)
    Eve_bits.append(p.measure(modes[mode]))
    Eve_prepared.append(p)

Then, modify Bob logic: Bob receives the photons sent by Eve.

In [22]:
# Bob receives the photons, assume no environmental noise
Bob_received = Eve_prepared.copy()
Eve_prepared.clear()

In [23]:
# Bob randomly selects modes to measure the photons.
Bob_modes = []
Bob_bits = []
for p in Bob_received:
    mode = mode_select()
    Bob_modes.append(mode)
    Bob_bits.append(p.measure(modes[mode]))

# Alice and Bob compare their sequence of modes and sift the list of keys. 
keys = []
for am,bm,ab,bb in zip(Alice_modes,Bob_modes,Alice_bits,Bob_bits):
    if (am == bm): # compared modes through a classical channel
        # assert ab == bb
        if ab == bb:
            keys.append(ab)

print ('Sifted keys:', len(keys))
print(keys)

Sifted keys: 399
[np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int32(1), np.int32(0), np.int32(0), np.int32(0), np.int32(1), np.int32(0), np.int32(1), np.int32(1), np.int32(0), np.int3

After running the modified python code, we can observe that the length of sifted key is about 375, and here is the reason. Eve randomly chooses modes to measure the photons, which means there are 50% of the choices of Eve are the same with ALice's, and the last 50% of photons will change the states. Consider Alice and Bob, Bob can choose the same modes with Alice's with the probability of 50%, so in the 50% photons whose state is not changed, there is 50% of probability Bob can get the right results, which is 250 photons. In the other 50% of photons whose state is changed by Eve, there is a probability of 50% Bob can choose the same modes with Alice's, which means these choices are not the same with Eve's, thus, there is a probability of 25% in total that Bob can get the same results with Alices's, which is 125 photons. So the final result of the simulation program is about 375.

### E2. Eve Improves Her Stategy

Consider Eve intercepts the photons transmitted from Alice to Bob in the BB84 protocol. Show that when Eve measures the photon polarization based on the two modes in the protocol, she has a 75% probability to measure $+1$ when Alice sends $+1$. 

Suppose Eve now measures photon polarization along a direction making an arbitrary angle $\zeta$ with the $x$ axis. Implement the new measurement method in Class Photon. Note that we do not need an elegant implementation, as the class was implemented by classical probability rules. Show that, for a certain range of $\zeta$, Eve can have a probability $P(\zeta) > 0.75$ to measure $+1$ when Alice sends $+1$. Plot $P(\zeta)$ as a function of $\zeta$. Along which $\zeta$ will Eve obtain the maximal probability? 

### Comments:

* What Eve really wants is that Alice and Bob should establish a key that Eve knows, so that she can secretly read the messages sent from Alice to Bob. _Eve will have success in preventing the construction of a key, but not in secretly learning a key that will be used by Alice and Bob._

* Why do we call this scheme quantum key distribution? After all, polarization is a classical concept. Which part of the BB84 depends crucially on the quantum concept and can be defeated in the classical world? __Quantum no-cloning theorem__: It is impossible to create an identical copy of an arbitrary unknown quantum state. We will come to its proof in Lecture 6.

* For additional security concerns, see, e.g., N. Luetkenhaus in _Lectures on Quantum Information_, edited by D. Bruss and G. Leuchs (Wiley-VCH, 2007). 

For the first problem, when Eve measures the photon polarization based on the two modes in the protocol and Alice sends $+1$, Eve has a $50\%$ probability to use the same protocol as Alice, and the probability of measuring $+1$ is $100\%$. For the other $50\%$ probability, the probability of measuring $+1$ is $50\%$. So the total probability $P=0.5\times 1+0.5\times 0.5=0.75$.

For the second problem, we implement the new measurement method in Class Photon first:

In [24]:
def measure_angle(self, zeta):
    ''' All information one can read is through measurement, which may change the state. '''        
    m_basis = [H * np.cos(zeta) + V * np.sin(zeta), H * np.sin(zeta) - V * np.cos(zeta)]
    if np.random.rand() < measurement_probability(self._state, m_basis[0]):
        self._state = m_basis[0]
        return 0
    else:
        self._state = m_basis[1]
        return 1
Photon.measure_angle = measure_angle

Then, we input the angle zeta and begin to simulate Eve's new stategy:

In [27]:
# Alice: prepare photons to be tranmitted.
Alice_modes = []
Alice_prepared = []
Alice_bits = [1] * nbits # Alice simply sends bit +1
for i in range(len(Alice_bits)):
    mode = mode_select()
    Alice_modes.append(mode)
    Alice_prepared.append(Photon(modes[mode][Alice_bits[i]]))

# Eve receives the photons, assume no environmental noise
Eve_received = Alice_prepared.copy()
Alice_prepared.clear()

# Eve measures photon polarization along a direction making an arbitrary angle zeta with the x-axis.
Eve_bits = []
zeta = float(input()) / 180 * np.acos(-1) # turn to deg
for p in Eve_received:
    Eve_bits.append(p.measure_angle(zeta))

# Eve checks how many +1 has she measured. 
keys = []
for ab,eb in zip(Alice_bits,Eve_bits):
    if ab == eb:
        keys.append(ab)

print ('Measured keys:', len(keys))

 22.5


Measured keys: 858


After running the modified python code, we can observe that when we input 0 or 45, the length of measured keys is about 750, same as the proof above. And when we input 22.5, the length of measured keys is about 854, which is longer than 750. The next thing is to derive $P(\zeta)$ as a function of $\zeta$.

When we use the angle $\zeta$, we have a $50\%$ probability to measure the photon used in mode I, and the probability of measuring $+1$ is $P_{I}(\zeta)=\cos^2(\zeta)=\frac{1+\cos(2\zeta)}{2}$. For the other $50\%$ probability, we have $P_{II}(\zeta)=\cos^2(\frac{\pi}{4}-\zeta)=\frac{1+\sin(2\zeta)}{2}$.

So we have the total probability $P(\zeta)=\frac{P_{I}(\zeta)+P_{II}(\zeta)}{2}=\frac{2+cos(2\zeta)+sin(2\zeta)}{4}=\frac{2+\sqrt{2}sin(2\zeta+\frac{\pi}{4})}{4}$.

<center>
<img src="Figure.png" width="600">
</center>

The figure above is $P(\zeta)$ as a function of $\zeta$. From the figure, we know that when $\zeta$ is between $0$ and $\frac{\pi}{4}$, Eve can have a probability $P(\zeta)>0.75$. In addition, when $\zeta=\frac{\pi}{8}$, Eve can have a maximal probability $P(\frac{\pi}{8})=\frac{2+\sqrt{2}}{4}\approx 0.854$.